In [ ]:
import os,sys
import re
import numpy as np
import pandas as pd
import scanpy as sc
from scanpy import AnnData
import mudata
from scipy.sparse import csr_matrix
import warnings
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import harmonypy as hm

os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'
sc.settings.verbosity = 3
warnings.filterwarnings("ignore")
plt.rcParams['pdf.fonttype']=42

In [ ]:
bgs5 = sc.read_h5ad("/u/project/cluo/rayirfan/cosmx/analyses/basalganglia/D_snm3/bgs5_snm3_3T_L2_L3.h5ad")

#3T --> bgs5

In [ ]:
bgs5

In [ ]:
bgs4 = sc.read_h5ad("/u/project/cluo/mbaig/cosmx/analyses/20251130_MB_basal_3/processed_data/20260106/20260107_bgs4_reseg_snm3_2T_L3.h5ad")

#2T --> bgs4 LGE and striatum only

In [ ]:
bgs4

In [ ]:
bgs5 = sc.read_h5ad("/u/project/cluo/mbaig/cosmx/analyses/20251130_MB_basal_3/processed_data/20260106/20260108_bgs5_filtered_snm3_3T_L3.h5ad")

#3T --> bgs5_filtered hat was made just now
bgs5

In [ ]:
#resegmented

bgs4 = sc.read_h5ad("/u/project/cluo/mbaig/cosmx/analyses/20251130_MB_basal_3/processed_data/20260106/20260107_bgs4_reseg_snm3_2T_L3.h5ad")

bgs4

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
from matplotlib.lines import Line2D

# Use your AnnData object
adata = bgs4  # or bgs4_lge_mge if that's the one you're using

# Get cluster annotation
clusters = adata.obs['adjusted_L3_transferred']
cluster_str = clusters.astype(str)

# Exclude 'uncertain' cells entirely
mask_known = cluster_str != 'uncertain'
cluster_str_filtered = cluster_str[mask_known]

# Get unique known clusters and sort nicely (numbers first, then others)
numeric = pd.to_numeric(clusters[mask_known], errors='coerce')
is_numeric = numeric.notna()
numeric_unique = np.sort(numeric[is_numeric].unique().astype(int))
non_numeric_unique = np.sort(clusters[mask_known & ~is_numeric].unique())

all_unique = [str(c) for c in numeric_unique] + [str(c) for c in non_numeric_unique]

# Use pre-defined colors if available (recommended!)
if ('adjusted_L3_transferred' in adata.obs and
    adata.obs['adjusted_L3_transferred'].dtype.name == 'category' and
    'adjusted_L3_transferred_colors' in adata.uns):

    categories = adata.obs['adjusted_L3_transferred'].cat.categories
    known_categories = [cat for cat in categories if str(cat) != 'uncertain']
    known_categories_str = [str(cat) for cat in known_categories]
    color_list = adata.uns['adjusted_L3_transferred_colors']
    # Match colors to known categories
    cat_to_color = dict(zip([str(cat) for cat in categories], color_list))
    cluster_colors = {clust: cat_to_color.get(clust, 'gray') for clust in all_unique}
    print("Using pre-defined colors from adata.uns")
else:
    # Fallback: generate palette
    palette = sns.color_palette('tab20', len(all_unique))
    cluster_colors = dict(zip(all_unique, palette))
    print("Using generated color palette")

# Coordinates (filtered to exclude uncertain)
if 'spatial' in adata.obsm:
    spatial_coords = adata.obsm['spatial'][mask_known]
    x_spatial, y_spatial = spatial_coords[:, 0], spatial_coords[:, 1]
else:
    x_spatial = adata.obs['CenterX_global_px'][mask_known]
    y_spatial = adata.obs['CenterY_global_px'][mask_known]

umap_coords = adata.obsm['X_umap'][mask_known]

# Create figure
fig, axes = plt.subplots(1, 2, figsize=(20, 10))

# Spatial plot
sns.scatterplot(x=x_spatial, y=y_spatial,
                hue=cluster_str_filtered,
                palette=cluster_colors,
                s=3, edgecolor=None, ax=axes[0], legend=False)

axes[0].set_title('Adjusted L3 Clusters - Spatial (uncertain excluded)')
axes[0].set_xlabel('X (px)')
axes[0].set_ylabel('Y (px)')
axes[0].invert_yaxis()
axes[0].set_aspect('equal')

# UMAP plot
sns.scatterplot(x=umap_coords[:, 0], y=umap_coords[:, 1],
                hue=cluster_str_filtered,
                palette=cluster_colors,
                s=2, edgecolor=None, ax=axes[1], legend=False)

axes[1].set_title('Adjusted L3 Clusters - UMAP (uncertain excluded)')
axes[1].set_xlabel('UMAP 1')
axes[1].set_ylabel('UMAP 2')

# Shared external legend
handles = [Line2D([0], [0], marker='o', color='w',
                  markerfacecolor=cluster_colors[clust],
                  markersize=10) for clust in all_unique]

fig.legend(handles=handles, labels=all_unique,
           title='Adjusted L3 Clusters',
           bbox_to_anchor=(1.02, 0.5), loc='center left', fontsize=10)

plt.tight_layout(rect=[0, 0, 0.88, 1])
plt.show()

#2T-->bgs4_reseg ('harmony_failed/uncertain' cluster hidden)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

adata = bgs4
categ = 'adjusted_L3_transferred'

# Get unique clusters
clusters = sorted(adata.obs[categ].unique())
n_clusters = len(clusters)

# Grid setup
ncols = 4
nrows = int(np.ceil(n_clusters / ncols))
figsize = (6 * ncols, 6 * nrows)

# Create figure
fig, axes = plt.subplots(nrows, ncols, figsize=figsize)
axes = axes.flatten() if n_clusters > 1 else [axes]

# Get spatial coordinates
spatial_coords = adata.obsm['spatial']

# Get colors for clusters
if f'{categ}_colors' in adata.uns:
    cluster_colors = dict(zip(adata.obs[categ].cat.categories, adata.uns[f'{categ}_colors']))
else:
    from matplotlib import cm
    cluster_colors = {c: cm.tab20(i % 20) for i, c in enumerate(clusters)}

grey_color = '#D3D3D3'
spot_size = 0.5

# Plot each cluster
for idx, cluster in enumerate(clusters):
    ax = axes[idx]

    # Create mask for this cluster
    mask = adata.obs[categ] == cluster

    # Plot grey background (all other cells)
    ax.scatter(
        spatial_coords[~mask, 0],
        spatial_coords[~mask, 1],
        c=grey_color,
        s=spot_size,
        alpha=0.5
    )

    # Plot this cluster in color
    ax.scatter(
        spatial_coords[mask, 0],
        spatial_coords[mask, 1],
        c=cluster_colors[cluster],
        s=spot_size,
        alpha=1.0
    )

    ax.set_title(f'Leiden {cluster}')
    ax.set_aspect('equal')
    ax.invert_yaxis()
    ax.axis('off')

# Remove empty subplots
for idx in range(n_clusters, len(axes)):
    fig.delaxes(axes[idx])

plt.tight_layout()
#plt.savefig('individual_clusters_spatial.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import scipy.sparse as sp

# Two separate heatmaps: marker gene expression across cell types
# - Titles: "2T  GW24 (bgs4)" and "3T  GW35 (bgs5)"
# - Z-score clipped to [-2, 2]
# - Cell counts shown on the right
# - Fully customizable: genes + separate cell type lists for each dataset

# 1. Explicitly define genes to show (in desired order)
custom_gene_order = [
    'DRD1',      # Direct MSN marker
    'DRD2',      # Indirect MSN marker

    'BACH2',     # Transcription factor (DRD1 subtype)
    'EPHA4',     # Eph receptor (DRD1/DRD2 subtype)
    'PENK',      # Enkephalin - classic D2-MSN marker (indirect pathway)
    'GAD1',      # GABAergic interneuron marker

    'PAX6',
    #'PDGFRA',
    'OLIG1',
    #'SOX2',
    #'NES',

    'OLIG2',     # Oligodendrocyte lineage
    'SOX10',     # Oligodendrocyte lineage
    'PDGFRA',     # Strong OPC and pericyte marker (careful overlap with PC)
    'MBP',        # Myelin basic protein - mature oligodendrocyte marker

    'ALDH1L1',   # Astrocyte marker
    'GFAP',      # Astrocyte marker
    'AQP4',      # Astrocyte marker

    'ESAM',      # Endothelial marker
    'FN1',       # Endothelial / pericyte
    'LUM',        # Lumican - top VLMC/fibroblast-like marker
    'DCN',        # Decorin - VLMC marker

    'P2RY12',    # Microglia marker
    'CX3CR1',     # Fractalkine receptor - microglia marker (complements P2RY12)
    #'SLC17A7',    # VGLUT1 - excitatory neuron marker (useful for any glutamatergic contamination or comparison)

]

# 2. Explicitly define cell types to KEEP for EACH dataset
# Edit these lists independently - only types in the list will appear in that heatmap
cell_types_bgs4 = [          # 2T  GW24
    'DRD1-BACH2',
    'DRD1-EPHA4',
    'DRD1-eccentric-CASZ1',
    'DRD2-BACH2',
    'DRD2-EPHA4',
    'eMSN',

    'GPC',
    'OPC',
    #'PAX6-TCx',
    'Astro',

    'EC',
    'PC',
    #'VLMC',
    'MGC-1',
    # 'uncertain',  # excluded
]

cell_types_bgs5 = [          # 3T  GW35
    'DRD1-BACH2',
    'DRD1-EPHA4',
    'DRD1-eccentric-CASZ1',
    'DRD2-BACH2',
    'DRD2-EPHA4',
    'DRD2-eccentric-CASZ1',
    'CABLES1',

    'GPC',
    'ODC',
    'OPC',
    'Astro',

    #'PVALB',
    #'SST',
    'EC',
    'PC',
    #'VLMC',
    'MGC-1',
    # 'uncertain',  # excluded
]

# 3. Filtering and area-based QC
label_col = 'adjusted_L3_transferred'
min_area_um2 = 10
um_per_pixel = 0.12028

bgs4_f = bgs4.copy()
bgs5_f = bgs5.copy()

# for adata in [bgs4_f, bgs5_f]:
#     nuc_area = adata.obs['NucArea'] * (um_per_pixel ** 2)
#     cell_area = adata.obs['Area'] * (um_per_pixel ** 2)
#     mask = (nuc_area >= min_area_um2) & (cell_area >= min_area_um2)
#     adata = adata[mask].copy()

# 4. Restrict to desired cell types for each dataset separately
bgs4_f = bgs4_f[bgs4_f.obs[label_col].isin(cell_types_bgs4)].copy()
bgs5_f = bgs5_f[bgs5_f.obs[label_col].isin(cell_types_bgs5)].copy()

# 5. Pseudobulk z-score function
def pseudobulk_zscore_fine(adata, genes, groupby=label_col):
    genes_use = [g for g in genes if g in adata.var_names]
    print(f"{adata.uns.get('name','adata')} - using {len(genes_use)}/{len(genes)} genes")

    if 'counts_downsampled' in adata.layers:
        X = adata[:, genes_use].layers['counts_downsampled']
    else:
        X = adata[:, genes_use].X

    if sp.issparse(X):
        X = X.toarray()

    df = pd.DataFrame(X, columns=genes_use)
    df[groupby] = adata.obs[groupby].values
    pb = df.groupby(groupby).mean()

    pb_z = (pb - pb.mean(axis=0)) / (pb.std(axis=0) + 1e-8)
    pb_z = pb_z.fillna(0)
    pb_z = pb_z.clip(lower=-2, upper=2)

    return pb_z, genes_use

# 6. Compute pseudobulk for each dataset
pb4, genes4 = pseudobulk_zscore_fine(bgs4_f, custom_gene_order)
pb5, genes5 = pseudobulk_zscore_fine(bgs5_f, custom_gene_order)

# Use only genes present in BOTH datasets (for visual alignment)
shared_genes = [g for g in custom_gene_order if g in genes4 and g in genes5]
pb4 = pb4.loc[:, shared_genes]
pb5 = pb5.loc[:, shared_genes]

# 6b. Reorder cell types in pseudobulk according to custom order
pb4 = pb4.reindex(cell_types_bgs4)
pb5 = pb5.reindex(cell_types_bgs5)

# 7. Cell counts
counts4 = bgs4_f.obs[label_col].value_counts().reindex(pb4.index).fillna(0).astype(int)
counts5 = bgs5_f.obs[label_col].value_counts().reindex(pb5.index).fillna(0).astype(int)

# 8. Plotting
fig = plt.figure(figsize=(14, (len(pb4) + len(pb5)) * 0.32 + 6))
gs = fig.add_gridspec(2, 1, hspace=0.2, height_ratios=[len(pb4), len(pb5)])

# Top: 2T  GW24
ax1 = fig.add_subplot(gs[0])
g1 = sns.heatmap(pb4, cmap='vlag', vmin=-2, vmax=2, center=0,
                 linewidths=0.5, linecolor='lightgray',
                 cbar_kws={"orientation": "horizontal", "pad": 0.2, "shrink": 0.6},  # <- increase pad
                 ax=ax1)
ax1.set_title("2T  GW24 (bgs4) - Selected Cell Types (pseudobulk z-scored, clipped [-2, 2])",
              fontsize=15, pad=12)
ax1.set_ylabel("Cell Type", fontsize=13)
ax1.set_xlabel("")
ax1.tick_params(axis='x', rotation=90, labelsize=11)
cbar1 = g1.collections[0].colorbar
cbar1.set_label('z-score', fontsize=11)

# Cell counts (right side)
for i, n in enumerate(counts4):
    ax1.text(len(shared_genes) + 0.25, i + 0.5, f"n={n:,}", va='center', ha='left', fontsize=9)
ax1.axvline(len(shared_genes), color='black', linewidth=1.2, linestyle='--')

# Bottom: 3T  GW35
ax2 = fig.add_subplot(gs[1])
g2 = sns.heatmap(pb5, cmap='vlag', vmin=-2, vmax=2, center=0,
                 linewidths=0.5, linecolor='lightgray',
                 cbar_kws={"orientation": "horizontal", "pad": 0.2, "shrink": 0.6},  # <- increase pad
                 ax=ax2)
ax2.set_title("3T  GW35 (bgs5) - Selected Cell Types (pseudobulk z-scored, clipped [-2, 2])",
              fontsize=15, pad=12)
ax2.set_ylabel("Cell Type", fontsize=13)
ax2.set_xlabel("Genes", fontsize=13)
ax2.tick_params(axis='x', rotation=90, labelsize=11)
cbar2 = g2.collections[0].colorbar
cbar2.set_label('z-score', fontsize=11)

# Cell counts (right side)
for i, n in enumerate(counts5):
    ax2.text(len(shared_genes) + 0.25, i + 0.5, f"n={n:,}", va='center', ha='left', fontsize=9)
ax2.axvline(len(shared_genes), color='black', linewidth=1.2, linestyle='--')

plt.suptitle("\nMarker Gene Expression Across Selected Cell Types\n2T  GW24 (top) vs 3T  GW35 (bottom)",
             fontsize=17, y=0.985)

plt.tight_layout(rect=[0, 0, 1, 0.99])
plt.show()

In [ ]:
# Imports
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import matplotlib as mpl
from matplotlib.ticker import ScalarFormatter

# Global style
mpl.rcParams.update({
    'font.size': 14,
    'axes.titlesize': 20,
    'axes.labelsize': 18,
    'xtick.labelsize': 14,
    'ytick.labelsize': 16,
    'legend.fontsize': 14,
})

# Parameters
min_area_um2 = 10
um_per_pixel = 0.12028
nuc_area_col = 'NucArea'
cell_area_col = 'Area'
label_col = 'adjusted_L3_transferred'
min_cells_per_dataset = 10

# Prepare data
df4 = bgs4.obs[[label_col, nuc_area_col, cell_area_col]].copy()
df4['Dataset'] = 'GW24'

df5 = bgs5.obs[[label_col, nuc_area_col, cell_area_col]].copy()
df5['Dataset'] = 'GW35'

combined = pd.concat([df4, df5], ignore_index=True)

# Drop bad labels
combined = combined.dropna(subset=[label_col])
combined = combined[~combined[label_col].isin(['?', 'uncertain'])]

# Convert to µm²
combined['NucArea_um2'] = combined[nuc_area_col] * (um_per_pixel ** 2)
combined['CellArea_um2'] = combined[cell_area_col] * (um_per_pixel ** 2)

# Area filters
combined = combined[
    (combined['NucArea_um2'] >= min_area_um2) &
    (combined['CellArea_um2'] >= min_area_um2)
]

# Keep only cell types with enough cells in BOTH datasets
counts = combined.groupby(['Dataset', label_col]).size().unstack(fill_value=0)

sufficient = (
    (counts.loc['GW24'] >= min_cells_per_dataset) &
    (counts.loc['GW35'] >= min_cells_per_dataset)
)

common_types = sorted(sufficient[sufficient].index.tolist())

print(f"Found {len(common_types)} cell types with ≥{min_cells_per_dataset} cells in both GW24 and GW35:")
print(common_types)

combined = combined[combined[label_col].isin(common_types)]
combined[label_col] = pd.Categorical(
    combined[label_col],
    categories=common_types,
    ordered=True
)

# Formatter for plain numbers on log axes
def plain_log_formatter(ax):
    formatter = ScalarFormatter()
    formatter.set_scientific(False)
    formatter.set_useOffset(False)
    ax.xaxis.set_major_formatter(formatter)
    ax.xaxis.set_minor_formatter(formatter)
    ax.tick_params(axis='x', which='minor', length=0)

# Plotting
fig, axes = plt.subplots(
    1, 2,
    figsize=(24, len(common_types) * 0.7 + 4),
    sharey=True
)

# Nuclear area
sns.barplot(
    data=combined,
    y=label_col,
    x='NucArea_um2',
    hue='Dataset',
    order=common_types,
    palette={'GW24': '#1f77b4', 'GW35': '#ff7f0e'},
    errorbar=('ci', 95),
    errwidth=2,
    capsize=0.1,
    ax=axes[0]
)

axes[0].set_title('Median Nuclear Area', fontsize=22)
axes[0].set_xlabel('Median Nuclear Area (µm²)')
axes[0].set_ylabel('Cell Type')
axes[0].set_xscale('log')
plain_log_formatter(axes[0])
axes[0].legend(title='Stage', loc='lower right')

# Cell area
sns.barplot(
    data=combined,
    y=label_col,
    x='CellArea_um2',
    hue='Dataset',
    order=common_types,
    palette={'GW24': '#1f77b4', 'GW35': '#ff7f0e'},
    errorbar=('ci', 95),
    errwidth=2,
    capsize=0.1,
    ax=axes[1]
)

axes[1].set_title('Median Cell Area', fontsize=22)
axes[1].set_xlabel('Median Cell Area (µm²)')
axes[1].set_ylabel('')
axes[1].set_xscale('log')
plain_log_formatter(axes[1])
axes[1].legend_.remove()

# Figure title
plt.suptitle(
    'Basal Ganglia L3 Cell Types Present in Both Stages\n'
    'Median Nuclear and Cell Areas with 95% CI (GW24 vs GW35)',
    fontsize=24,
    y=0.98
)

plt.tight_layout(rect=[0, 0, 1, 0.94])
plt.show()

# Summary table
medians = (
    combined
    .groupby(['Dataset', label_col])[['NucArea_um2', 'CellArea_um2']]
    .median()
    .reset_index()
)

summary_medians = medians.pivot(
    index=label_col,
    columns='Dataset',
    values=['NucArea_um2', 'CellArea_um2']
)

summary_medians.columns = [
    f"{metric}_{stage}" for metric, stage in summary_medians.columns
]

summary_medians = summary_medians.round(1).sort_index()

print("\nSummary Table: Median Areas (µm²) - Cell Types with ≥10 Cells in Both Stages")
print(summary_medians)

In [ ]:
# Imports
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import matplotlib as mpl
from matplotlib.ticker import ScalarFormatter

# Style
mpl.rcParams.update({
    'font.size': 14,
    'axes.titlesize': 20,
    'axes.labelsize': 18,
    'xtick.labelsize': 14,
    'ytick.labelsize': 16,
})

# Parameters
min_area_um2 = 10
um_per_pixel = 0.12028
nuc_area_col = 'NucArea'
cell_area_col = 'Area'
label_col = 'adjusted_L3_transferred'
min_cells_per_dataset = 10

# Prepare data
df4 = bgs4.obs[[label_col, nuc_area_col, cell_area_col]].copy()
df4['Dataset'] = 'GW24'

df5 = bgs5.obs[[label_col, nuc_area_col, cell_area_col]].copy()
df5['Dataset'] = 'GW35'

combined = pd.concat([df4, df5], ignore_index=True)

combined = combined.dropna(subset=[label_col])
combined = combined[~combined[label_col].isin(['?', 'uncertain'])]

combined['NucArea_um2'] = combined[nuc_area_col] * (um_per_pixel ** 2)
combined['CellArea_um2'] = combined[cell_area_col] * (um_per_pixel ** 2)

combined = combined[
    (combined['NucArea_um2'] >= min_area_um2) &
    (combined['CellArea_um2'] >= min_area_um2)
]

# Keep shared cell types
counts = combined.groupby(['Dataset', label_col]).size().unstack(fill_value=0)

sufficient = (
    (counts.loc['GW24'] >= min_cells_per_dataset) &
    (counts.loc['GW35'] >= min_cells_per_dataset)
)

common_types = sorted(sufficient[sufficient].index.tolist())
combined = combined[combined[label_col].isin(common_types)]

combined[label_col] = pd.Categorical(
    combined[label_col],
    categories=common_types,
    ordered=True
)

# Plain formatter for log axes
def plain_log_formatter(ax):
    formatter = ScalarFormatter()
    formatter.set_scientific(False)
    formatter.set_useOffset(False)
    ax.xaxis.set_major_formatter(formatter)

# Plot
fig, axes = plt.subplots(
    1, 2,
    figsize=(24, len(common_types) * 0.7 + 4),
    sharey=True
)

sns.barplot(
    data=combined,
    y=label_col,
    x='NucArea_um2',
    hue='Dataset',
    estimator=np.median,
    errorbar=('ci', 95),
    order=common_types,
    ax=axes[0]
)
axes[0].set_title('Median Nuclear Area')
axes[0].set_xlabel('Median Nuclear Area (µm²)')
axes[0].set_ylabel('Cell Type')
axes[0].set_xscale('log')
plain_log_formatter(axes[0])

sns.barplot(
    data=combined,
    y=label_col,
    x='CellArea_um2',
    hue='Dataset',
    estimator=np.median,
    errorbar=('ci', 95),
    order=common_types,
    ax=axes[1]
)
axes[1].set_title('Median Cell Area')
axes[1].set_xlabel('Median Cell Area (µm²)')
axes[1].set_ylabel('')
axes[1].set_xscale('log')
plain_log_formatter(axes[1])
axes[1].legend_.remove()

plt.suptitle(
    'Basal Ganglia L3 Cell Types\n'
    'Median Nuclear and Cell Areas with 95% Bootstrap CI',
    fontsize=24,
    y=0.98
)

plt.tight_layout(rect=[0, 0, 1, 0.94])
plt.show()

In [ ]:
# Imports
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import matplotlib as mpl

# Style
mpl.rcParams.update({
    'font.size': 14,
    'axes.titlesize': 20,
    'axes.labelsize': 18,
    'xtick.labelsize': 14,
    'ytick.labelsize': 16,
    'legend.fontsize': 14,
})

# Parameters
min_area_um2 = 10
um_per_pixel = 0.12028
nuc_area_col = 'NucArea'
cell_area_col = 'Area'
label_col = 'adjusted_L3_transferred'
n_bootstrap = 1000
ci_level = 95
min_cells_per_dataset = 10

# Prepare data
df4 = bgs4.obs[[label_col, nuc_area_col, cell_area_col]].copy()
df4['Dataset'] = 'GW24 (2T)'

df5 = bgs5.obs[[label_col, nuc_area_col, cell_area_col]].copy()
df5['Dataset'] = 'GW35 (3T)'

combined = pd.concat([df4, df5], ignore_index=True)

combined = combined.dropna(subset=[label_col])
combined = combined[~combined[label_col].isin(['?', 'uncertain'])]

combined['NucArea_um2'] = combined[nuc_area_col] * (um_per_pixel ** 2)
combined['CellArea_um2'] = combined[cell_area_col] * (um_per_pixel ** 2)

combined = combined[
    (combined['NucArea_um2'] >= min_area_um2) &
    (combined['CellArea_um2'] >= min_area_um2)
]

# Filter cell types present in BOTH datasets
counts = combined.groupby(['Dataset', label_col]).size().unstack(fill_value=0)

sufficient = (
    (counts.loc['GW24 (2T)'] >= min_cells_per_dataset) &
    (counts.loc['GW35 (3T)'] >= min_cells_per_dataset)
)

# Alphabetical list
common_types = sorted(sufficient[sufficient].index.tolist())

if len(common_types) == 0:
    raise ValueError("No cell types meet the minimum cell count requirement.")

combined = combined[combined[label_col].isin(common_types)]

print(f"Found {len(common_types)} cell types with ≥{min_cells_per_dataset} cells in both datasets:")
print(common_types)

# Bootstrap median CI
def bootstrap_median_ci(data, n_bootstrap=1000, ci_level=95):
    boot = np.random.choice(data, size=(n_bootstrap, len(data)), replace=True)
    medians = np.median(boot, axis=1)
    lo = np.percentile(medians, (100 - ci_level) / 2)
    hi = np.percentile(medians, 100 - (100 - ci_level) / 2)
    return lo, hi

# Compute statistics
rows = []

for ct in common_types:
    gw24 = combined[(combined['Dataset'] == 'GW24 (2T)') & (combined[label_col] == ct)]
    gw35 = combined[(combined['Dataset'] == 'GW35 (3T)') & (combined[label_col] == ct)]

    m24n = np.median(gw24['NucArea_um2'])
    m35n = np.median(gw35['NucArea_um2'])
    m24c = np.median(gw24['CellArea_um2'])
    m35c = np.median(gw35['CellArea_um2'])

    n24_lo, n24_hi = bootstrap_median_ci(gw24['NucArea_um2'], n_bootstrap, ci_level)
    n35_lo, n35_hi = bootstrap_median_ci(gw35['NucArea_um2'], n_bootstrap, ci_level)
    c24_lo, c24_hi = bootstrap_median_ci(gw24['CellArea_um2'], n_bootstrap, ci_level)
    c35_lo, c35_hi = bootstrap_median_ci(gw35['CellArea_um2'], n_bootstrap, ci_level)

    rows.append({
        'CellType': ct,
        'Log2FC_Nuclear': np.log2(m35n / m24n),
        'Nuclear_low_err': np.log2(m35n / m24n) - np.log2(n35_lo / n24_hi),
        'Nuclear_up_err': np.log2(n35_hi / n24_lo) - np.log2(m35n / m24n),
        'Log2FC_Cell': np.log2(m35c / m24c),
        'Cell_low_err': np.log2(m35c / m24c) - np.log2(c35_lo / c24_hi),
        'Cell_up_err': np.log2(c35_hi / c24_lo) - np.log2(m35c / m24c),
        'Median_Nuc_GW24': m24n,
        'Median_Nuc_GW35': m35n,
        'Median_Cell_GW24': m24c,
        'Median_Cell_GW35': m35c
    })

df = pd.DataFrame(rows).sort_values('CellType')

#  THIS IS THE KEY LINE
order = df['CellType'].tolist()[::-1]   # reverse alphabetical for barh()

# Plot
fig, axes = plt.subplots(1, 2, figsize=(20, len(order) * 0.65 + 5), sharey=True)

axes[0].barh(
    order,
    df['Log2FC_Nuclear'],
    xerr=[df['Nuclear_low_err'], df['Nuclear_up_err']],
    color='#2ca02c',
    capsize=5,
    error_kw={'elinewidth': 2, 'capthick': 2}
)
axes[0].axvline(0, color='gray', linestyle='--')
axes[0].set_title('Nuclear Area')
axes[0].set_xlabel('Log₂ Fold Change (GW35 / GW24)')
axes[0].set_ylabel('Cell Type')

axes[1].barh(
    order,
    df['Log2FC_Cell'],
    xerr=[df['Cell_low_err'], df['Cell_up_err']],
    color='#d62728',
    capsize=5,
    error_kw={'elinewidth': 2, 'capthick': 2}
)
axes[1].axvline(0, color='gray', linestyle='--')
axes[1].set_title('Cell Area')
axes[1].set_xlabel('Log₂ Fold Change (GW35 / GW24)')
axes[1].set_ylabel('')

plt.suptitle(
    'Median Size Changes (GW35 vs GW24)\n'
    'Log₂ Fold Change of Medians with 95% Bootstrapped Confidence Intervals',
    fontsize=24,
    y=0.98
)

plt.tight_layout(rect=[0, 0, 1, 0.94])
plt.show()

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import matplotlib as mpl

# STYLE
mpl.rcParams.update({
    'font.size': 14,
    'axes.titlesize': 20,
    'axes.labelsize': 18,
    'xtick.labelsize': 14,
    'ytick.labelsize': 16,
    'legend.fontsize': 14,
})

# PARAMETERS
min_area_um2 = 10
um_per_pixel = 0.12028
nuc_area_col = 'NucArea'
cell_area_col = 'Area'
label_col = 'adjusted_L3_transferred'

# PREPARE DATA
df4 = bgs4.obs[[label_col, nuc_area_col, cell_area_col]].copy()
df4['Dataset'] = 'GW24 (2T)'  # Corrected label
df5 = bgs5.obs[[label_col, nuc_area_col, cell_area_col]].copy()
df5['Dataset'] = 'GW35 (3T)'

combined = pd.concat([df4, df5], ignore_index=True)

# Remove uncertain / unknown labels
combined = combined.dropna(subset=[label_col])
combined = combined[combined[label_col] != '?']
combined = combined[combined[label_col] != 'uncertain']

# Convert to µm²
combined['NucArea_um2'] = combined[nuc_area_col] * (um_per_pixel ** 2)
combined['CellArea_um2'] = combined[cell_area_col] * (um_per_pixel ** 2)

# Filter small objects
combined = combined[(combined['NucArea_um2'] >= min_area_um2) & (combined['CellArea_um2'] >= min_area_um2)]

# MEDIAN VALUES & LOG2 FOLD CHANGE
medians = combined.groupby(['Dataset', label_col])[ ['NucArea_um2', 'CellArea_um2'] ].median()

gw24_medians = medians.loc['GW24 (2T)']
gw35_medians = medians.loc['GW35 (3T)']

# Only keep cell types present in both datasets
common_types = gw24_medians.index.intersection(gw35_medians.index)
gw24_medians = gw24_medians.loc[common_types]
gw35_medians = gw35_medians.loc[common_types]

# Compute log2 fold change (GW35 / GW24)
fc_nuc = np.log2(gw35_medians['NucArea_um2'] / gw24_medians['NucArea_um2'])
fc_cell = np.log2(gw35_medians['CellArea_um2'] / gw24_medians['CellArea_um2'])

# Alphabetical order
order = sorted(common_types)

# PLOTTING
fig, axes = plt.subplots(1, 2, figsize=(20, len(order) * 0.65 + 5), sharey=True)

# Nuclear Area Log2 Fold Change (median-based)
sns.barplot(x=fc_nuc[order], y=order, color='#2ca02c', ax=axes[0])
axes[0].set_title('Nuclear Area', fontsize=20)
axes[0].set_xlabel('Log₂ Fold Change (median)')
axes[0].set_ylabel('Cell Type')
axes[0].axvline(0, color='gray', linewidth=1, linestyle='--')

# Cell Area Log2 Fold Change (median-based)
sns.barplot(x=fc_cell[order], y=order, color='#d62728', ax=axes[1])
axes[1].set_title('Cell Area', fontsize=20)
axes[1].set_xlabel('Log₂ Fold Change (median)')
axes[1].set_ylabel('')
axes[1].axvline(0, color='gray', linewidth=1, linestyle='--')

# Main title
plt.suptitle('Median Size Changes\n'
             'Log₂ Fold Change of Median Areas (GW35 ÷ GW24)\n'
             '<---GW24 larger          GW35 larger--->',
             fontsize=24, y=1.00)

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

# SUMMARY TABLE
summary = pd.DataFrame({
    'Median_Nuc_GW24': gw24_medians['NucArea_um2'].round(1),
    'Median_Nuc_GW35': gw35_medians['NucArea_um2'].round(1),
    'Log2FC_Nuclear': fc_nuc.round(3),
    'Median_Cell_GW24': gw24_medians['CellArea_um2'].round(1),
    'Median_Cell_GW35': gw35_medians['CellArea_um2'].round(1),
    'Log2FC_Cell': fc_cell.round(3),
}).loc[order]

print("\nSummary Table (median-based, alphabetical order, common cell types only):")
print(summary)

#reseg, there are no negative log fold changes so everything is bigger in GW35

In [ ]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.ticker import ScalarFormatter

# Style
mpl.rcParams.update({
    'font.size': 14,
    'axes.titlesize': 20,
    'axes.labelsize': 18,
    'xtick.labelsize': 14,
    'ytick.labelsize': 16,
})

# Parameters
min_area_um2 = 10
um_per_pixel = 0.12028
nuc_area_col = 'NucArea'
cell_area_col = 'Area'
label_col = 'adjusted_L3_transferred'
min_cells_per_dataset = 10
n_bootstrap = 1000
ci_level = 95

# Prepare data (SINGLE FILTERING PATH)
df4 = bgs4.obs[[label_col, nuc_area_col, cell_area_col]].copy()
df4['Dataset'] = 'GW24'

df5 = bgs5.obs[[label_col, nuc_area_col, cell_area_col]].copy()
df5['Dataset'] = 'GW35'

combined = pd.concat([df4, df5], ignore_index=True)

combined = combined.dropna(subset=[label_col])
combined = combined[~combined[label_col].isin(['?', 'uncertain'])]

combined['NucArea_um2'] = combined[nuc_area_col] * (um_per_pixel ** 2)
combined['CellArea_um2'] = combined[cell_area_col] * (um_per_pixel ** 2)

combined = combined[
    (combined['NucArea_um2'] >= min_area_um2) &
    (combined['CellArea_um2'] >= min_area_um2)
]

# Keep only shared cell types
counts = combined.groupby(['Dataset', label_col]).size().unstack(fill_value=0)

shared = (
    (counts.loc['GW24'] >= min_cells_per_dataset) &
    (counts.loc['GW35'] >= min_cells_per_dataset)
)

cell_types = sorted(shared[shared].index.tolist())
combined = combined[combined[label_col].isin(cell_types)]

print(f"Using {len(cell_types)} shared cell types:")
print(cell_types)

# Bootstrap utilities (SINGLE DEFINITION)
def bootstrap_median_ci(x, n_bootstrap=1000, ci_level=95):
    x = np.asarray(x)
    boots = np.median(
        np.random.choice(x, size=(n_bootstrap, len(x)), replace=True),
        axis=1
    )
    lo = np.percentile(boots, (100 - ci_level) / 2)
    hi = np.percentile(boots, 100 - (100 - ci_level) / 2)
    return np.median(x), lo, hi

def bootstrap_log2fc(x, y, n_bootstrap=1000, ci_level=95):
    x = np.asarray(x)
    y = np.asarray(y)
    boots = []

    for _ in range(n_bootstrap):
        xb = np.random.choice(x, size=len(x), replace=True)
        yb = np.random.choice(y, size=len(y), replace=True)
        boots.append(np.log2(np.median(yb) / np.median(xb)))

    boots = np.array(boots)
    center = np.median(boots)
    lo = np.percentile(boots, (100 - ci_level) / 2)
    hi = np.percentile(boots, 100 - (100 - ci_level) / 2)
    return center, lo, hi

# Compute ALL statistics ONCE
rows = []

for ct in cell_types:
    gw24 = combined[(combined['Dataset'] == 'GW24') & (combined[label_col] == ct)]
    gw35 = combined[(combined['Dataset'] == 'GW35') & (combined[label_col] == ct)]

    # Absolute medians
    m24_n, n24_lo, n24_hi = bootstrap_median_ci(gw24['NucArea_um2'], n_bootstrap, ci_level)
    m35_n, n35_lo, n35_hi = bootstrap_median_ci(gw35['NucArea_um2'], n_bootstrap, ci_level)

    m24_c, c24_lo, c24_hi = bootstrap_median_ci(gw24['CellArea_um2'], n_bootstrap, ci_level)
    m35_c, c35_lo, c35_hi = bootstrap_median_ci(gw35['CellArea_um2'], n_bootstrap, ci_level)

    # Log2FC
    log2fc_n, log2fc_n_lo, log2fc_n_hi = bootstrap_log2fc(
        gw24['NucArea_um2'], gw35['NucArea_um2'], n_bootstrap, ci_level
    )

    log2fc_c, log2fc_c_lo, log2fc_c_hi = bootstrap_log2fc(
        gw24['CellArea_um2'], gw35['CellArea_um2'], n_bootstrap, ci_level
    )

    rows.append({
        'CellType': ct,

        'GW24_Nuc': m24_n, 'GW24_Nuc_lo': n24_lo, 'GW24_Nuc_hi': n24_hi,
        'GW35_Nuc': m35_n, 'GW35_Nuc_lo': n35_lo, 'GW35_Nuc_hi': n35_hi,

        'GW24_Cell': m24_c, 'GW24_Cell_lo': c24_lo, 'GW24_Cell_hi': c24_hi,
        'GW35_Cell': m35_c, 'GW35_Cell_lo': c35_lo, 'GW35_Cell_hi': c35_hi,

        'Log2FC_Nuc': log2fc_n,
        'Log2FC_Nuc_lo': log2fc_n - log2fc_n_lo,
        'Log2FC_Nuc_hi': log2fc_n_hi - log2fc_n,

        'Log2FC_Cell': log2fc_c,
        'Log2FC_Cell_lo': log2fc_c - log2fc_c_lo,
        'Log2FC_Cell_hi': log2fc_c_hi - log2fc_c,
    })

stats = pd.DataFrame(rows).sort_values('CellType')
order = stats['CellType'].tolist()[::-1]

# Plot ABSOLUTE + LOG2FC (GUARANTEED CONSISTENT)
fig, axes = plt.subplots(2, 2, figsize=(26, len(order) * 0.6 + 6), sharey=True)

def plain_log(ax):
    f = ScalarFormatter()
    f.set_scientific(False)
    ax.xaxis.set_major_formatter(f)

# Absolute Nuclear
axes[0,0].barh(order, stats['GW24_Nuc'], xerr=[stats['GW24_Nuc']-stats['GW24_Nuc_lo'],
                                              stats['GW24_Nuc_hi']-stats['GW24_Nuc']],
               color='#1f77b4', label='GW24')
axes[0,0].barh(order, stats['GW35_Nuc'], left=stats['GW24_Nuc'],
               color='#ff7f0e', label='GW35')
axes[0,0].set_xscale('log')
plain_log(axes[0,0])
axes[0,0].set_title('Median Nuclear Area')

# Absolute Cell
axes[0,1].barh(order, stats['GW24_Cell'], color='#1f77b4')
axes[0,1].barh(order, stats['GW35_Cell'], left=stats['GW24_Cell'], color='#ff7f0e')
axes[0,1].set_xscale('log')
plain_log(axes[0,1])
axes[0,1].set_title('Median Cell Area')

# Log2FC Nuclear
axes[1,0].barh(order, stats['Log2FC_Nuc'],
               xerr=[stats['Log2FC_Nuc_lo'], stats['Log2FC_Nuc_hi']],
               color='#2ca02c', capsize=4)
axes[1,0].axvline(0, color='gray', linestyle='--')
axes[1,0].set_title('Log₂FC Nuclear (GW35 / GW24)')

# Log2FC Cell
axes[1,1].barh(order, stats['Log2FC_Cell'],
               xerr=[stats['Log2FC_Cell_lo'], stats['Log2FC_Cell_hi']],
               color='#d62728', capsize=4)
axes[1,1].axvline(0, color='gray', linestyle='--')
axes[1,1].set_title('Log₂FC Cell (GW35 / GW24)')

plt.suptitle(
    'Basal Ganglia L3 Cell Size Changes\n'
    'Single-Pipeline Median Statistics (Absolute & Log₂FC)',
    fontsize=26,
    y=0.98
)

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

In [ ]:
# Imports
import pandas as pd
import numpy as np

# Parameters
min_area_um2 = 10
um_per_pixel = 0.12028
nuc_area_col = 'NucArea'
cell_area_col = 'Area'
label_col = 'adjusted_L3_transferred'
min_cells_per_dataset = 10
n_bootstrap = 1000
ci_level = 95

# Prepare data
df4 = bgs4.obs[[label_col, nuc_area_col, cell_area_col]].copy()
df4['Dataset'] = 'GW24'

df5 = bgs5.obs[[label_col, nuc_area_col, cell_area_col]].copy()
df5['Dataset'] = 'GW35'

combined = pd.concat([df4, df5], ignore_index=True)

combined = combined.dropna(subset=[label_col])
combined = combined[~combined[label_col].isin(['?', 'uncertain'])]

combined['NucArea_um2'] = combined[nuc_area_col] * (um_per_pixel ** 2)
combined['CellArea_um2'] = combined[cell_area_col] * (um_per_pixel ** 2)

combined = combined[
    (combined['NucArea_um2'] >= min_area_um2) &
    (combined['CellArea_um2'] >= min_area_um2)
]

# Keep only shared cell types
counts = combined.groupby(['Dataset', label_col]).size().unstack(fill_value=0)

shared = (
    (counts.loc['GW24'] >= min_cells_per_dataset) &
    (counts.loc['GW35'] >= min_cells_per_dataset)
)

cell_types = sorted(shared[shared].index.tolist())
combined = combined[combined[label_col].isin(cell_types)]

print(f"Using {len(cell_types)} shared cell types:")
print(cell_types)

# Bootstrap median and log2FC functions
def bootstrap_median(x, n_bootstrap=1000, ci_level=95):
    x = np.asarray(x)
    boots = np.median(np.random.choice(x, size=(n_bootstrap, len(x)), replace=True), axis=1)
    lo = np.percentile(boots, (100 - ci_level)/2)
    hi = np.percentile(boots, 100 - (100 - ci_level)/2)
    return np.median(x), lo, hi

def bootstrap_log2fc(x, y, n_bootstrap=1000, ci_level=95):
    x = np.asarray(x)
    y = np.asarray(y)
    boots = []
    for _ in range(n_bootstrap):
        xb = np.random.choice(x, size=len(x), replace=True)
        yb = np.random.choice(y, size=len(y), replace=True)
        boots.append(np.log2(np.median(yb) / np.median(xb)))
    boots = np.array(boots)
    center = np.median(boots)
    lo = np.percentile(boots, (100 - ci_level)/2)
    hi = np.percentile(boots, 100 - (100 - ci_level)/2)
    return center, lo, hi

# Compute medians and log2FC
rows = []

for ct in cell_types:
    gw24 = combined[(combined['Dataset'] == 'GW24') & (combined[label_col] == ct)]
    gw35 = combined[(combined['Dataset'] == 'GW35') & (combined[label_col] == ct)]

    # Absolute medians
    m24_n, n24_lo, n24_hi = bootstrap_median(gw24['NucArea_um2'], n_bootstrap, ci_level)
    m35_n, n35_lo, n35_hi = bootstrap_median(gw35['NucArea_um2'], n_bootstrap, ci_level)

    m24_c, c24_lo, c24_hi = bootstrap_median(gw24['CellArea_um2'], n_bootstrap, ci_level)
    m35_c, c35_lo, c35_hi = bootstrap_median(gw35['CellArea_um2'], n_bootstrap, ci_level)

    # Log2FC
    log2fc_n, lo_n, hi_n = bootstrap_log2fc(gw24['NucArea_um2'], gw35['NucArea_um2'], n_bootstrap, ci_level)
    log2fc_c, lo_c, hi_c = bootstrap_log2fc(gw24['CellArea_um2'], gw35['CellArea_um2'], n_bootstrap, ci_level)

    rows.append({
        'CellType': ct,

        # Absolute medians
        'Median_Nuc_GW24': m24_n,
        'Median_Nuc_GW35': m35_n,
        'Median_Cell_GW24': m24_c,
        'Median_Cell_GW35': m35_c,

        # Log2FC
        'Log2FC_Nuclear': log2fc_n,
        'Log2FC_Cell': log2fc_c,
    })

stats = pd.DataFrame(rows).set_index('CellType').sort_index()

# Summary tables
# Table 1: Median areas
median_areas = stats[['Median_Nuc_GW24', 'Median_Nuc_GW35', 'Median_Cell_GW24', 'Median_Cell_GW35']]
median_areas = median_areas.round(1)
print("\nSummary Table: Median Areas (µm²) - Common Cell Types Only")
print(median_areas)

# Table 2: Median + log2FC
summary_table = stats[['Median_Nuc_GW24', 'Median_Nuc_GW35', 'Log2FC_Nuclear',
                       'Median_Cell_GW24', 'Median_Cell_GW35', 'Log2FC_Cell']]
summary_table = summary_table.round(3)
print("\nSummary Table (median-based, alphabetical order, common cell types only):")
print(summary_table)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib as mpl

# Style
mpl.rcParams.update({
    'font.size': 14,
    'axes.titlesize': 20,
    'axes.labelsize': 18,
    'xtick.labelsize': 14,
    'ytick.labelsize': 14,
})

# Parameters
label_col = 'adjusted_L3_transferred'
um_per_pixel = 0.12028
min_nuc_area_um2 = 10
min_cells = 20

# Helper: build nuclear size dataframe
def build_df(adata, stage):
    nuc_area_um2 = adata.obs['NucArea'] * (um_per_pixel ** 2)
    mask = nuc_area_um2 >= min_nuc_area_um2

    return pd.DataFrame({
        'cell_type': adata.obs.loc[mask, label_col],
        'nuc_area_um2': nuc_area_um2[mask],
        'stage': stage
    })

df4 = build_df(bgs4, 'GW24')
df5 = build_df(bgs5, 'GW35')

# Restrict to cell types starting with "D"
df4 = df4[df4['cell_type'].str.startswith('D')]
df5 = df5[df5['cell_type'].str.startswith('D')]

# Keep only cell types present at BOTH timepoints
cts4 = df4['cell_type'].value_counts()
cts5 = df5['cell_type'].value_counts()

common_types = sorted([
    ct for ct in set(cts4.index) & set(cts5.index)
    if cts4[ct] >= min_cells and cts5[ct] >= min_cells
])

df4 = df4[df4['cell_type'].isin(common_types)]
df5 = df5[df5['cell_type'].isin(common_types)]

print(f"Included D* cell types: {common_types}")

# Compute median nuclear size per cell type and stage
summary = (
    pd.concat([df4, df5])
    .groupby(['cell_type', 'stage'])['nuc_area_um2']
    .median()
    .reset_index()
)

# Ensure correct time ordering
summary['stage'] = pd.Categorical(
    summary['stage'],
    categories=['GW24', 'GW35'],
    ordered=True
)

# Plot: two-point time series
fig, ax = plt.subplots(figsize=(10, 7))

# Manual color control
# DRD1 = blues, DRD2 = reds
palette = {
    # DRD1
    'DRD1-BACH2': '#27ab44',                 # blue
    'DRD1-EPHA4': '#08306b',                 # lighter blue
    'DRD1-eccentric-CASZ1': '#10c4e8',       # dark blue

    # DRD2
    'DRD2-BACH2': '#929494',                 # red
    'DRD2-EPHA4': '#a15747',                 # light red
    #'DRD2-eccentric-CASZ1': '#7f0000',       # dark red
}

# Manual label offsets (µm²)
# Positive = move label UP, Negative = move label DOWN
label_offsets = {
    # Example (edit freely):
    'DRD1-BACH2': -0.5,
    'DRD1-EPHA4': +0.5,
    # 'DRD1-eccentric-CASZ1': 10,
    # 'DRD2-BACH2': -6,
}

for ct in common_types:
    sub = summary[summary['cell_type'] == ct]

    ax.plot(
        sub['stage'],
        sub['nuc_area_um2'],
        marker='o',
        lw=2.5,
        color=palette[ct]
    )

    # Label on the right (GW35) with manual offset
    y_end = sub[sub['stage'] == 'GW35']['nuc_area_um2'].values[0]
    y_offset = label_offsets.get(ct, 0)

    ax.text(
        1.02,
        y_end + y_offset,
        ct,
        color=palette[ct],
        fontsize=13,
        va='center',
        transform=ax.get_yaxis_transform()
    )

# Axis formatting
ax.set_title('Nuclear Size Changes Across Development')
ax.set_xlabel('Developmental Stage')
ax.set_ylabel('Median Nuclear Area (µm²)')
ax.set_xlim(-0.1, 1.1)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib as mpl

# Style
mpl.rcParams.update({
    'font.size': 14,
    'axes.titlesize': 20,
    'axes.labelsize': 18,
    'xtick.labelsize': 14,
    'ytick.labelsize': 14,
})

# Parameters
label_col = 'adjusted_L3_transferred'
um_per_pixel = 0.12028
min_nuc_area_um2 = 10
min_cells = 20

target_celltypes = ['OPC', 'GPC', 'Astro']   #  only these three

# Helper: build nuclear size dataframe
def build_df(adata, stage):
    nuc_area_um2 = adata.obs['NucArea'] * (um_per_pixel ** 2)
    mask = nuc_area_um2 >= min_nuc_area_um2
    return pd.DataFrame({
        'cell_type': adata.obs.loc[mask, label_col],
        'nuc_area_um2': nuc_area_um2[mask],
        'stage': stage
    })

df4 = build_df(bgs4, 'GW24')
df5 = build_df(bgs5, 'GW35')

# Restrict to our three target cell types only
df4 = df4[df4['cell_type'].isin(target_celltypes)]
df5 = df5[df5['cell_type'].isin(target_celltypes)]

# Keep only cell types present at BOTH timepoints with enough cells
cts4 = df4['cell_type'].value_counts()
cts5 = df5['cell_type'].value_counts()

common_types = sorted([
    ct for ct in target_celltypes
    if ct in cts4.index and ct in cts5.index
    and cts4[ct] >= min_cells and cts5[ct] >= min_cells
])

df4 = df4[df4['cell_type'].isin(common_types)]
df5 = df5[df5['cell_type'].isin(common_types)]

print(f"Included cell types: {common_types}")

# Compute median nuclear size per cell type and stage
summary = (
    pd.concat([df4, df5])
    .groupby(['cell_type', 'stage'])['nuc_area_um2']
    .median()
    .reset_index()
)

# Ensure correct time ordering
summary['stage'] = pd.Categorical(
    summary['stage'],
    categories=['GW24', 'GW35'],
    ordered=True
)

# Plot: two-point time series
fig, ax = plt.subplots(figsize=(8, 6.5))

# Suggested color palette for OPC / GPC / Astros
# Feel free to change the colors!
palette = {
    'OPC':   '#d95f02',     # orange-brown
    'GPC':   '#1b9e77',     # teal/green
    'Astro': '#7570b3',    # purple
}

# Manual label vertical offsets (µm²) - tweak as needed
# Positive = move label UP, Negative = move label DOWN
label_offsets = {
    'OPC':    0.0,
    'GPC':    1.2,
    'Astro': -1.0,
}

for ct in common_types:
    sub = summary[summary['cell_type'] == ct]
    ax.plot(
        sub['stage'],
        sub['nuc_area_um2'],
        marker='o',
        markersize=9,
        lw=2.8,
        color=palette.get(ct, 'gray')
    )

    # Label on the right (GW35) with manual offset
    y_end = sub[sub['stage'] == 'GW35']['nuc_area_um2'].values[0]
    y_offset = label_offsets.get(ct, 0)
    ax.text(
        1.02,
        y_end + y_offset,
        ct,
        color=palette.get(ct, 'gray'),
        fontsize=13,
        fontweight='medium',
        va='center',
        transform=ax.get_yaxis_transform()
    )

# Axis formatting
ax.set_title('Nuclear Size Changes Across Development\n(OPC, GPC, Astrocytes)',
             pad=15)
ax.set_xlabel('Developmental Stage')
ax.set_ylabel('Median Nuclear Area (µm²)')
ax.set_xlim(-0.15, 1.15)
ax.grid(axis='y', alpha=0.25, linestyle='--')
ax.tick_params(axis='both', which='major', length=5)

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib as mpl

# Style
mpl.rcParams.update({
    'font.size': 14,
    'axes.titlesize': 20,
    'axes.labelsize': 18,
    'xtick.labelsize': 14,
    'ytick.labelsize': 14,
})

# Parameters
label_col = 'adjusted_L3_transferred'
um_per_pixel = 0.12028
min_nuc_area_um2 = 10
min_cells = 20

target_celltypes = ['EC', 'MGC-1', 'PC']   #  only these three

# Helper: build nuclear size dataframe
def build_df(adata, stage):
    nuc_area_um2 = adata.obs['NucArea'] * (um_per_pixel ** 2)
    mask = nuc_area_um2 >= min_nuc_area_um2
    return pd.DataFrame({
        'cell_type': adata.obs.loc[mask, label_col],
        'nuc_area_um2': nuc_area_um2[mask],
        'stage': stage
    })

df4 = build_df(bgs4, 'GW24')
df5 = build_df(bgs5, 'GW35')

# Restrict to our three target cell types only
df4 = df4[df4['cell_type'].isin(target_celltypes)]
df5 = df5[df5['cell_type'].isin(target_celltypes)]

# Keep only cell types present at BOTH timepoints with enough cells
cts4 = df4['cell_type'].value_counts()
cts5 = df5['cell_type'].value_counts()

common_types = sorted([
    ct for ct in target_celltypes
    if ct in cts4.index and ct in cts5.index
    and cts4[ct] >= min_cells and cts5[ct] >= min_cells
])

df4 = df4[df4['cell_type'].isin(common_types)]
df5 = df5[df5['cell_type'].isin(common_types)]

print(f"Included cell types: {common_types}")

# Compute median nuclear size per cell type and stage
summary = (
    pd.concat([df4, df5])
    .groupby(['cell_type', 'stage'])['nuc_area_um2']
    .median()
    .reset_index()
)

# Ensure correct time ordering
summary['stage'] = pd.Categorical(
    summary['stage'],
    categories=['GW24', 'GW35'],
    ordered=True
)

# Plot: two-point time series
fig, ax = plt.subplots(figsize=(8, 6.5))

# Suggested color palette for vascular/glial cell types
# Feel free to adjust!
palette = {
    'EC':     '#e41a1c',    # strong red - classic for endothelial
    'MGC-1':  '#377eb8',    # blue - often used for Müller glia
    'PC':     '#4daf4a',    # green - common for pericytes
}

# Manual label vertical offsets (µm²) - tweak these to avoid overlap
# Positive = move label UP, Negative = move label DOWN
label_offsets = {
    'EC':     0.0,
    'MGC-1':  1.5,
    'PC':    -1.2,
}

for ct in common_types:
    sub = summary[summary['cell_type'] == ct]
    ax.plot(
        sub['stage'],
        sub['nuc_area_um2'],
        marker='o',
        markersize=9,
        lw=2.8,
        color=palette.get(ct, 'gray')
    )

    # Label on the right (GW35) with manual offset
    y_end = sub[sub['stage'] == 'GW35']['nuc_area_um2'].values[0]
    y_offset = label_offsets.get(ct, 0)
    ax.text(
        1.02,
        y_end + y_offset,
        ct,
        color=palette.get(ct, 'gray'),
        fontsize=13,
        fontweight='medium',
        va='center',
        transform=ax.get_yaxis_transform()
    )

# Axis formatting
ax.set_title('Nuclear Size Changes Across Development\n(EC, MGC-1, Pericytes)',
             pad=15)
ax.set_xlabel('Developmental Stage')
ax.set_ylabel('Median Nuclear Area (µm²)')
ax.set_xlim(-0.15, 1.15)
ax.grid(axis='y', alpha=0.25, linestyle='--')
ax.tick_params(axis='both', which='major', length=5)

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib as mpl

# Style settings
mpl.rcParams.update({
    'font.size': 13,
    'axes.titlesize': 16,
    'axes.labelsize': 14,
    'xtick.labelsize': 12,
    'ytick.labelsize': 12,
    'legend.fontsize': 11,
})

# Global parameters
label_col = 'adjusted_L3_transferred'
um_per_pixel = 0.12028
min_nuc_area_um2 = 10
min_cells = 20

# All groups we want to plot
groups = {
    'DRD': ['DRD1-BACH2', 'DRD1-EPHA4', 'DRD1-eccentric-CASZ1',
            'DRD2-BACH2', 'DRD2-EPHA4'],  # add more if needed

    'Glia': ['OPC', 'GPC', 'Astro'],

    'Vascular': ['EC', 'MGC-1', 'PC']
}

# Suggested color palettes for each group
palettes = {
    'DRD': {
        'DRD1-BACH2': '#27ab44',
        'DRD1-EPHA4': '#08306b',
        'DRD1-eccentric-CASZ1': '#10c4e8',
        'DRD2-BACH2': '#d95f02',
        'DRD2-EPHA4': '#a15747',
    },
    'Glia': {
        'OPC':   '#d95f02',
        'GPC':   '#1b9e77',
        'Astro': '#7570b3',
    },
    'Vascular': {
        'EC':     '#e41a1c',
        'MGC-1':  '#377eb8',
        'PC':     '#4daf4a',
    }
}

# Label vertical offsets (µm²) - tune per group / cell type as needed
label_offsets = {
    'DRD': {
        'DRD1-BACH2': 0.0,
        'DRD1-EPHA4': 1.2,
        'DRD1-eccentric-CASZ1': -0.8,
        'DRD2-BACH2': 0.5,
        'DRD2-EPHA4': -1.0,
    },
    'Glia': {
        'OPC':    0.0,
        'GPC':    1.4,
        'Astro': -1.2,
    },
    'Vascular': {
        'EC':     0.0,
        'MGC-1':  1.3,
        'PC':    -1.0,
    }
}

# Helper functions
def build_df(adata, stage):
    nuc_area_um2 = adata.obs['NucArea'] * (um_per_pixel ** 2)
    mask = nuc_area_um2 >= min_nuc_area_um2
    return pd.DataFrame({
        'cell_type': adata.obs.loc[mask, label_col],
        'nuc_area_um2': nuc_area_um2[mask],
        'stage': stage
    })

# Build raw data
df4 = build_df(bgs4, 'GW24')
df5 = build_df(bgs5, 'GW35')
df_all = pd.concat([df4, df5], ignore_index=True)

# Prepare summary statistics with 95% CI for ALL cell types
summary = (
    df_all
    .groupby(['cell_type', 'stage'])['nuc_area_um2']
    .agg(['median', 'count', 'std'])
    .reset_index()
)

# Compute 95% CI (approximate using normal distribution)
# For small n you might prefer t-distribution, but median + normal is very common
summary['sem'] = summary['std'] / np.sqrt(summary['count'])
summary['ci_low'] = summary['median'] - 1.96 * summary['sem']
summary['ci_high'] = summary['median'] + 1.96 * summary['sem']

# Keep only types that appear in at least one group
all_desired = set(sum(groups.values(), []))
summary = summary[summary['cell_type'].isin(all_desired)]

# Order stages
summary['stage'] = pd.Categorical(summary['stage'],
                                 categories=['GW24', 'GW35'], ordered=True)
summary = summary.sort_values(['cell_type', 'stage'])

# Plot - 3 subplots
fig, axes = plt.subplots(1, 3, figsize=(15, 6), sharey=True,
                         gridspec_kw={'wspace': 0.08})

for ax, (group_name, celltypes) in zip(axes, groups.items()):
    palette = palettes[group_name]
    offsets = label_offsets[group_name]

    present_types = [ct for ct in celltypes if ct in summary['cell_type'].values]

    for ct in present_types:
        sub = summary[summary['cell_type'] == ct].sort_values('stage')
        if len(sub) < 2:
            continue

        x = np.arange(len(sub))  # 0 = GW24, 1 = GW35
        y = sub['median']
        yerr_low = y - sub['ci_low']
        yerr_high = sub['ci_high'] - y

        ax.errorbar(x, y, yerr=[yerr_low, yerr_high],
                    fmt='o-', capsize=5, capthick=1.5, elinewidth=1.8,
                    color=palette.get(ct, 'gray'), lw=2.2, markersize=8,
                    zorder=10)

        # Label at right side (GW35)
        if 'GW35' in sub['stage'].values:
            y_end = sub[sub['stage'] == 'GW35']['median'].values[0]
            offset = offsets.get(ct, 0)
            ax.text(1.05, y_end + offset, ct,
                    color=palette.get(ct, 'gray'),
                    fontsize=11.5, va='center',
                    transform=ax.get_yaxis_transform(),
                    fontweight='medium')

    # Formatting
    ax.set_title(f'{group_name} cell types', pad=10)
    ax.set_xticks([0, 1])
    ax.set_xticklabels(['GW24', 'GW35'])
    ax.grid(axis='y', alpha=0.25, linestyle='--')

    # Only leftmost plot gets y-label
    if ax == axes[0]:
        ax.set_ylabel('Median Nuclear Area (µm²)')

    ax.set_xlim(-0.3, 1.3)

# Overall figure title
fig.suptitle('Nuclear Size Changes Across Development + 95% CI',
             fontsize=18, y=1.02)

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib as mpl

# Very large font settings (optimized for small screenshot size)
mpl.rcParams.update({
    'font.size': 18,
    'axes.titlesize': 26,
    'axes.labelsize': 22,
    'xtick.labelsize': 20,
    'ytick.labelsize': 20,
    'legend.fontsize': 18,
})

# Global parameters
label_col = 'adjusted_L3_transferred'
um_per_pixel = 0.12028
min_nuc_area_um2 = 10
min_cells = 20

# Groups - MGC-1 moved to Glia
groups = {
    'DRD subtypes': ['DRD1-BACH2', 'DRD1-EPHA4', 'DRD1-eccentric-CASZ1',
                     'DRD2-BACH2', 'DRD2-EPHA4'],

    'Glia': ['OPC', 'GPC', 'Astro', 'MGC-1'],           #  MGC-1 moved here + Astros  Astro

    'Vascular': ['EC', 'PC']
}

# Color palettes (updated for Astro & MGC-1)
palettes = {
    'DRD subtypes': {
        'DRD1-BACH2': '#27ab44',
        'DRD1-EPHA4': '#08306b',
        'DRD1-eccentric-CASZ1': '#10c4e8',
        'DRD2-BACH2': '#d95f02',
        'DRD2-EPHA4': '#a15747',
    },
    'Glia': {
        'OPC':    '#d95f02',     # orange
        'GPC':    '#1b9e77',     # teal
        'Astro':  '#7570b3',     # purple
        'MGC-1':  '#e7298a',     # magenta/pink - distinct for microglia
    },
    'Vascular': {
        'EC': '#e41a1c',         # red
        'PC': '#4daf4a',         # green
    }
}

# Label vertical offsets - will probably need tuning after seeing real data
label_offsets = {
    'DRD subtypes': {
        'DRD1-BACH2': -1.2,
        'DRD1-EPHA4': 2.0,
        'DRD1-eccentric-CASZ1': -1.5,
        'DRD2-BACH2': -1,
        'DRD2-EPHA4': -3,
    },
    'Glia': {
        'OPC':    0.0,
        'GPC':    0,
        'Astro':  0,
        'MGC-1':  0,
    },
    'Vascular': {
        'EC':     0.0,
        'PC':    -0.5,
    }
}

# Helper: build dataframe
def build_df(adata, stage):
    nuc_area_um2 = adata.obs['NucArea'] * (um_per_pixel ** 2)
    mask = nuc_area_um2 >= min_nuc_area_um2
    df = pd.DataFrame({
        'cell_type': adata.obs.loc[mask, label_col],
        'nuc_area_um2': nuc_area_um2[mask],
        'stage': stage
    })
    # Quick rename Astros  Astro
    df['cell_type'] = df['cell_type'].replace('Astros', 'Astro')
    return df

df4 = build_df(bgs4, 'GW24')
df5 = build_df(bgs5, 'GW35')
df_all = pd.concat([df4, df5], ignore_index=True)

# Summary statistics with 95% CI
summary = (
    df_all
    .groupby(['cell_type', 'stage'])['nuc_area_um2']
    .agg(['median', 'count', 'std'])
    .reset_index()
)

summary['sem'] = summary['std'] / np.sqrt(summary['count'])
summary['ci_low']  = summary['median'] - 1.96 * summary['sem']
summary['ci_high'] = summary['median'] + 1.96 * summary['sem']

# Keep only desired types
all_desired = set(sum(groups.values(), []))
summary = summary[summary['cell_type'].isin(all_desired)]

summary['stage'] = pd.Categorical(summary['stage'],
                                 categories=['GW24', 'GW35'], ordered=True)
summary = summary.sort_values(['cell_type', 'stage'])

# Plot - VERTICAL layout, big fonts
fig, axes = plt.subplots(3, 1, figsize=(9, 16), sharex=True,
                         gridspec_kw={'hspace': 0.35})

for ax, (group_name, celltypes) in zip(axes, groups.items()):
    palette = palettes[group_name]
    offsets = label_offsets[group_name]

    present_types = [ct for ct in celltypes if ct in summary['cell_type'].values]

    for ct in present_types:
        sub = summary[summary['cell_type'] == ct].sort_values('stage')
        if len(sub) < 2:
            continue

        x = np.arange(len(sub))
        y = sub['median'].values
        yerr_low = y - sub['ci_low'].values
        yerr_high = sub['ci_high'].values - y

        ax.errorbar(x, y, yerr=[yerr_low, yerr_high],
                    fmt='o-', capsize=6, capthick=2, elinewidth=2.2,
                    color=palette.get(ct, 'gray'), lw=3.2, markersize=10,
                    zorder=10)

        # Big label on the right (GW35)
        if 'GW35' in sub['stage'].values:
            y_end = sub[sub['stage'] == 'GW35']['median'].values[0]
            offset = offsets.get(ct, 0)
            ax.text(1.04, y_end + offset, ct,
                    color=palette.get(ct, 'gray'),
                    fontsize=19, va='center',
                    transform=ax.get_yaxis_transform(),
                    fontweight='bold')

    # Formatting
    ax.set_title(group_name, pad=18, fontweight='bold')
    ax.set_xticks([0, 1])
    ax.set_xticklabels(['GW24', 'GW35'], fontsize=22)
    ax.grid(axis='y', alpha=0.3, linestyle='--', linewidth=1.2)
    ax.set_ylabel('Median Nuclear Area (µm²)', fontsize=18)
    ax.set_xlim(-0.3, 1.3)
    ax.tick_params(axis='both', which='major', length=6, width=1.3)

# Overall figure title (optional - comment out if you don't want it)
# fig.suptitle('Nuclear Size Changes + 95% CI Across Development',
#              fontsize=30, fontweight='bold', y=0.98)

plt.tight_layout()
plt.show()

In [ ]:
from pathlib import Path
import matplotlib as mpl

# Illustrator-friendly SVG settings
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
mpl.rcParams['svg.fonttype'] = 'none'   # keep text editable

# Output directory
outdir = Path("20260114 - figs")
outdir.mkdir(exist_ok=True)

# Loop over groups and save each as its own SVG
for group_name, celltypes in groups.items():

    fig, ax = plt.subplots(
        1, 1,
        figsize=(5.8, 4.2),   # tuned for single-panel vertical plots
    )

    # Tight margins (matches your composite figure style)
    plt.subplots_adjust(left=0.18, right=0.82, top=0.90, bottom=0.16)

    palette = palettes[group_name]
    offsets = label_offsets[group_name]
    present_types = [ct for ct in celltypes if ct in summary['cell_type'].values]

    # Plot each cell type
    for ct in present_types:
        sub = summary[summary['cell_type'] == ct].sort_values('stage')
        if len(sub) < 2:
            continue

        x = np.arange(len(sub))
        y = sub['median'].values
        yerr_low = y - sub['ci_low'].values
        yerr_high = sub['ci_high'].values - y

        ax.errorbar(
            x, y,
            yerr=[yerr_low, yerr_high],
            fmt='o-',
            capsize=6,
            capthick=2,
            elinewidth=2.2,
            color=palette.get(ct, 'gray'),
            lw=3.2,
            markersize=10,
            zorder=10
        )

        # Label at GW35
        if 'GW35' in sub['stage'].values:
            y_end = sub[sub['stage'] == 'GW35']['median'].values[0]
            offset = offsets.get(ct, 0)
            ax.text(
                1.04, y_end + offset, ct,
                color=palette.get(ct, 'gray'),
                fontsize=19,
                va='center',
                transform=ax.get_yaxis_transform(),
                fontweight='bold'
            )

    # Formatting
    ax.set_title(group_name, pad=8, fontweight='bold')
    ax.set_xticks([0, 1])
    ax.set_xticklabels(['GW24', 'GW35'], fontsize=22)
    ax.set_ylabel('Median Nuclear Area (µm²)', fontsize=16)
    ax.set_xlim(-0.3, 1.3)
    ax.grid(axis='y', alpha=0.3, linestyle='--', linewidth=1.2)
    ax.tick_params(axis='both', which='major', length=6, width=1.3)

    # Special annotations ONLY for MSN subtypes
    if group_name == 'MSN subtypes':
        ax.axvline(
            x=1.5,
            color='gray',
            linestyle=':',
            linewidth=2.2,
            alpha=0.65,
            zorder=5
        )

        # Striosome label
        ax.text(
            0.12, 0.67, 'striosome',
            transform=ax.transAxes,
            ha='center', va='bottom',
            fontsize=15, fontweight='bold',
            rotation=90,
            rotation_mode='anchor',
            bbox=dict(facecolor='white', edgecolor='none', pad=0, alpha=0.92)
        )

        ax.plot(
            [0.14, 0.14], [0.47, 0.85],
            color='black',
            linewidth=4.5,
            transform=ax.transAxes,
            clip_on=False
        )

    # Save SVG
    fname = group_name.replace(' ', '_').lower()
    fig.savefig(
        outdir / f"nuclear_area_{fname}.svg",
        bbox_inches="tight"
    )

    plt.close(fig)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import matplotlib as mpl

# Illustrator-friendly PDF settings (editable text)
# + Arial font
mpl.rcParams.update({
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
    'svg.fonttype': 'none',   # harmless to keep
    'font.family': 'Arial',
    'font.sans-serif': ['Arial'],
})

# Output: current notebook working directory
outdir = Path.cwd()

# Loop over groups and save each as its own PDF
for group_name, celltypes in groups.items():

    fig, ax = plt.subplots(
        1, 1,
        figsize=(5.8, 4.2),   # tuned for single-panel vertical plots
    )

    # Tight margins (matches your composite figure style)
    plt.subplots_adjust(left=0.18, right=0.82, top=0.90, bottom=0.16)

    palette = palettes[group_name]
    offsets = label_offsets[group_name]
    present_types = [ct for ct in celltypes if ct in summary['cell_type'].values]

    # Plot each cell type
    for ct in present_types:
        sub = summary[summary['cell_type'] == ct].sort_values('stage')
        if len(sub) < 2:
            continue

        x = np.arange(len(sub))
        y = sub['median'].values
        yerr_low = y - sub['ci_low'].values
        yerr_high = sub['ci_high'].values - y

        ax.errorbar(
            x, y,
            yerr=[yerr_low, yerr_high],
            fmt='o-',
            capsize=6,
            capthick=2,
            elinewidth=2.2,
            color=palette.get(ct, 'gray'),
            lw=3.2,
            markersize=10,
            zorder=10
        )

        # Label at GW35
        if 'GW35' in sub['stage'].values:
            y_end = sub[sub['stage'] == 'GW35']['median'].values[0]
            offset = offsets.get(ct, 0)
            ax.text(
                1.04, y_end + offset, ct,
                color=palette.get(ct, 'gray'),
                fontsize=19,
                va='center',
                transform=ax.get_yaxis_transform(),
                #fontweight='bold'
            )

    # Formatting
    ax.set_title(group_name, pad=8)#, fontweight='bold')
    ax.set_xticks([0, 1])
    ax.set_xticklabels(['GW24', 'GW35'], fontsize=22)
    ax.set_ylabel('Median Nuclear Area (µm²)', fontsize=16)
    ax.set_xlim(-0.3, 1.3)
    ax.grid(axis='y', alpha=0.3, linestyle='--', linewidth=1.2)
    ax.tick_params(axis='both', which='major', length=6, width=1.3)

    # Special annotations ONLY for MSN subtypes
    if group_name == 'MSN subtypes':
        ax.axvline(
            x=1.5,
            color='gray',
            linestyle=':',
            linewidth=2.2,
            alpha=0.65,
            zorder=5
        )

        # Striosome label
        ax.text(
            0.12, 0.67, 'striosome',
            transform=ax.transAxes,
            ha='center', va='bottom',
            fontsize=15,
            rotation=90,
            rotation_mode='anchor',
            bbox=dict(facecolor='white', edgecolor='none', pad=0, alpha=0.92)
        )

        ax.plot(
            [0.14, 0.14], [0.47, 0.85],
            color='black',
            linewidth=4.5,
            transform=ax.transAxes,
            clip_on=False
        )

    # Save PDF (current directory)
    fname = group_name.replace(' ', '_').lower()
    outpath = outdir / f"20260115_nuclear_area_{fname}.pdf"

    fig.savefig(
        outpath,
        format="pdf",
        bbox_inches="tight"
    )

    plt.close(fig)

print(f"Saved {len(groups)} PDFs to: {outdir}")

In [ ]:
import numpy as np
import pandas as pd

# Print nuclear area summary table (MEDIAN + 95% CI)
# Columns: GW24 median, GW24 CI_low, GW24 CI_high, GW35 median, GW35 CI_low, GW35 CI_high
# Assumes you already created `summary` earlier.
# Expected columns in `summary`: ['cell_type','stage','median','ci_low','ci_high']

required = {'cell_type', 'stage', 'median', 'ci_low', 'ci_high'}
missing = required - set(summary.columns)
if missing:
    raise ValueError(f"`summary` is missing required columns: {missing}")

df = summary[['cell_type', 'stage', 'median', 'ci_low', 'ci_high']].copy()
df['stage'] = df['stage'].astype(str)

# Split by stage
gw24 = df[df['stage'].str.contains('GW24', case=False, na=False)].copy()
gw35 = df[df['stage'].str.contains('GW35', case=False, na=False)].copy()

# Rename for wide format
gw24 = gw24.rename(columns={
    'median': 'GW24_median',
    'ci_low': 'GW24_ci_low',
    'ci_high': 'GW24_ci_high'
})
gw35 = gw35.rename(columns={
    'median': 'GW35_median',
    'ci_low': 'GW35_ci_low',
    'ci_high': 'GW35_ci_high'
})

# Keep one row per cell_type per stage (if duplicates exist, take first)
gw24 = gw24.sort_values('cell_type').drop_duplicates('cell_type')[
    ['cell_type', 'GW24_median', 'GW24_ci_low', 'GW24_ci_high']
]
gw35 = gw35.sort_values('cell_type').drop_duplicates('cell_type')[
    ['cell_type', 'GW35_median', 'GW35_ci_low', 'GW35_ci_high']
]

# Merge + order
wide = pd.merge(gw24, gw35, on='cell_type', how='outer').sort_values('cell_type')

pd.set_option('display.max_rows', 500)
pd.set_option('display.width', 200)

print("\nNuclear area summary (median + 95% CI):")
print(wide.to_string(index=False))